# PreprocessingNotebook này dùng để tiền xử lý dữ liệu trước khi xây dựng Data Warehouse / Iceberg Cube / Classification.**Mục tiêu:**- Chuẩn hóa kiểu dữ liệu (dimension vs measure)- Kiểm tra & xử lý outlier (BMI, PhysHlth, MentHlth) bằng capping/winsorize (tùy chọn)- Kiểm tra imbalance và đề xuất cách xử lý (class_weight/SMOTE)- Chia train/test + chuẩn hóa (scaling) cho các feature dạng số

In [1]:
# ---- Path setup: để import được code trong thư mục src/ ----import sysfrom pathlib import PathPROJECT_ROOT = Path.cwd()# Nếu đang chạy với cwd=notebooks/ thì project root sẽ là parentif not (PROJECT_ROOT / 'src' / 'diabetes_dw_prediction').exists():    PROJECT_ROOT = PROJECT_ROOT.parentsys.path.insert(0, str(PROJECT_ROOT))print('Project root:', PROJECT_ROOT)

Project root: d:\DataMing\Dm_Final\Diabetes-DW-Prediction

In [5]:
import importlibimport numpy as npimport pandas as pd# Optional: SMOTE (cần pip install imbalanced-learn)try:    from imblearn.over_sampling import SMOTE  # noqa: F401    IMBLEARN_AVAILABLE = Trueexcept Exception:    IMBLEARN_AVAILABLE = Falseimport src.diabetes_dw_prediction.data_loading as data_loadingimport src.diabetes_dw_prediction.preprocessing as pp# Reload để tránh dùng module cũ trong kernel sau khi bạn sửa file trong src/importlib.reload(data_loading)importlib.reload(pp)

<module 'src.diabetes_dw_prediction.preprocessing' from 'd:\\DataMing\\Dm_Final\\Diabetes-DW-Prediction\\src\\diabetes_dw_prediction\\preprocessing.py'>

In [6]:
# Load datasetdf = data_loading.load_and_prepare_brfss()print(df.shape)df.head()

[*] Dữ liệu đã có sẵn trong máy, bỏ qua bước tải.[*] Đang đọc file vào Pandas từ: dataset\diabetes_binary_health_indicators_BRFSS2015.csv-> Thành công! Kích thước dữ liệu: 253,680 dòng.(253680, 22)

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## 1) Chuẩn hóa kiểu dữ liệu (dimension vs measure)- **Dimension rời rạc/ordinal**: Age, Sex, Income, Education, GenHlth, HighBP, HighChol, Smoker, PhysActivity, ...- **Measure**: BMI, PhysHlth, MentHlth (dạng số/đếm ngày)

In [ ]:
# Chọn nhóm feature (dimension vs measure)fg = pp.default_feature_groups(target='Diabetes_binary')TARGET = fg.targetDIM_COLS = fg.dimsMEASURE_COLS = fg.measuresmissing = [c for c in [TARGET, *DIM_COLS, *MEASURE_COLS] if c not in df.columns]print('Missing columns:', missing)print('Target distribution:', df[TARGET].value_counts(normalize=True).rename('ratio'))

In [ ]:
# Ép kiểu dữ liệu theo nhóm featuredf_pp = pp.coerce_dimension_measure_dtypes(    df,    dims=DIM_COLS,    measures=MEASURE_COLS,    target=TARGET,    copy=True, )df_pp[DIM_COLS + MEASURE_COLS + [TARGET]].dtypes

## 2) Outlier check + capping/winsorize (tùy chọn)Kiểm tra bằng quantile (ví dụ 1% và 99%). Nếu muốn ổn định mô hình, có thể clip về các ngưỡng này.

In [ ]:
# Tính ngưỡng outlier theo quantile (p01, p99) cho các measurebounds = pp.compute_bounds_for_columns(df_pp, cols=MEASURE_COLS, low_q=0.01, high_q=0.99)pd.DataFrame(bounds, index=['p01', 'p99']).T

In [ ]:
APPLY_WINSORIZE = Trueif APPLY_WINSORIZE:    df_pp = pp.winsorize_clip(df_pp, bounds=bounds, copy=False)df_pp[MEASURE_COLS].describe().T

## 3) Imbalance handling (classification)- Nếu imbalance cao, ưu tiên **class_weight='balanced'** cho Logistic Regression / SVM.- Nếu cần, dùng **SMOTE** (cần `imbalanced-learn`).

In [ ]:
counts = df_pp[TARGET].value_counts(dropna=False)ratio = df_pp[TARGET].value_counts(normalize=True, dropna=False)display(pd.DataFrame({'count': counts, 'ratio': ratio}).sort_index())print('IMBLEARN_AVAILABLE:', IMBLEARN_AVAILABLE)if not IMBLEARN_AVAILABLE:    print('Nếu muốn dùng SMOTE: pip install imbalanced-learn')

## 4) Split train/test + scaling- Stratify theo nhãn để giữ tỷ lệ lớp.- Chỉ scale **MEASURE_COLS** (BMI/PhysHlth/MentHlth). Dimension ordinal/binary giữ nguyên.

In [ ]:
feature_cols = [c for c in (DIM_COLS + MEASURE_COLS) if c in df_pp.columns]X = df_pp[feature_cols].copy()y = df_pp[TARGET].astype('int64')X_train, X_test, y_train, y_test = pp.train_test_split_stratified(    X, y,    test_size=0.2,    random_state=42, )print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)print('Train label ratio:', y_train.value_counts(normalize=True))

In [ ]:
# Preprocessor: scale measure cols, pass-through dimspreprocessor = pp.build_preprocessor(    feature_cols=feature_cols,    numeric_cols=MEASURE_COLS,    scaler='standard', )X_train_ready = preprocessor.fit_transform(X_train)X_test_ready = preprocessor.transform(X_test)print('X_train_ready:', X_train_ready.shape)print('X_test_ready :', X_test_ready.shape)

## 5) (Tuỳ chọn) Lưu data đã tiền xử lýBạn có thể lưu file vào `data/processed/` để dùng lại cho DW/cube/model mà không cần chạy lại.

In [ ]:
from src.diabetes_dw_prediction.config import get_pathspaths = get_paths(PROJECT_ROOT)paths.data_processed_dir.mkdir(parents=True, exist_ok=True)parquet_path = paths.data_processed_dir / 'brfss_preprocessed.parquet'csv_path = paths.data_processed_dir / 'brfss_preprocessed.csv'try:    df_pp.to_parquet(parquet_path, index=False)    print('Saved:', parquet_path)except Exception as e:    # Nếu thiếu pyarrow/fastparquet thì fallback sang CSV    print('Không lưu được Parquet (thiếu engine). Fallback CSV. Error:', e)    df_pp.to_csv(csv_path, index=False)    print('Saved:', csv_path)